# Robot Arm Trajectory: Two Bases, Two Experiences

A robot arm joint sweeps from **0° to 90°** over $t \in [0,1]$.  
The trajectory is a degree-3 polynomial:

$$\theta(t) = c_0 + c_1 t + c_2 t^2 + c_3 t^3$$

### The motion basis $\{1,\ t,\ S(t)\}$

Define the **smooth-step** curve $S(t) = t^2(3-2t)$: starts flat, ends flat, goes $0 \to 1$.

$$\theta(t) = \underbrace{\alpha}_{\text{misalignment}} \cdot 1 + \underbrace{\beta}_{\text{drift}} \cdot t + \underbrace{\gamma}_{\text{swing}} \cdot S(t)$$

Each knob controls **exactly one** physical property.  
Expanding $S(t) = 3t^2 - 2t^3$ gives $c_2 = 3\gamma,\ c_3 = -2\gamma$, i.e. $2c_2 + 3c_3 = 0$.

> **Challenge:** using only the **monomial sliders**, change swing from 90° to 60° without breaking the S-curve.  
> You must move $c_2$ **and** $c_3$ together in ratio $3:-2$.  
> With the motion basis: just move $\gamma$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

t_arr = np.linspace(0, 1, 300)

def S(t):
    # smooth-step: 0->1, zero slope at both ends
    return t**2 * (3 - 2*t)

# ── sliders ────────────────────────────────────────────────────────────
W    = '360px'
ST_m = {'description_width': '130px'}
ST_c = {'description_width': '40px'}

sl_alpha = widgets.FloatSlider(value=0,   min=-30,  max=30,  step=0.5,
    description='α  misalign',  style=ST_m, layout=widgets.Layout(width=W))
sl_beta  = widgets.FloatSlider(value=0,   min=-60,  max=60,  step=0.5,
    description='β  drift',     style=ST_m, layout=widgets.Layout(width=W))
sl_gamma = widgets.FloatSlider(value=90,  min=-20,  max=180, step=1,
    description='γ  swing [°]', style=ST_m, layout=widgets.Layout(width=W))

sl_c0 = widgets.FloatSlider(value=0,    min=-30,  max=30,  step=0.5,
    description='c₀', style=ST_c, layout=widgets.Layout(width=W))
sl_c1 = widgets.FloatSlider(value=0,    min=-60,  max=60,  step=0.5,
    description='c₁', style=ST_c, layout=widgets.Layout(width=W))
sl_c2 = widgets.FloatSlider(value=270,  min=-500, max=500, step=1,
    description='c₂', style=ST_c, layout=widgets.Layout(width=W))
sl_c3 = widgets.FloatSlider(value=-180, min=-400, max=400, step=1,
    description='c₃', style=ST_c, layout=widgets.Layout(width=W))

sl_time = widgets.FloatSlider(
    value=0.5, min=0, max=1, step=0.01,
    description='time  t',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='600px'))

residual_html = widgets.HTML()
out           = widgets.Output()
_busy         = False


def get_theta():
    c0, c1, c2, c3 = sl_c0.value, sl_c1.value, sl_c2.value, sl_c3.value
    return c0 + c1*t_arr + c2*t_arr**2 + c3*t_arr**3


def draw(theta, time_val):
    theta_ideal = 90 * S(t_arr)
    ok  = abs(2*sl_c2.value + 3*sl_c3.value) < 5
    col = 'steelblue' if ok else 'tomato'

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(12, 4.5),
        gridspec_kw={'width_ratios': [1.5, 1]})

    # left: theta(t) trajectory
    ax1.plot(t_arr, theta_ideal, 'k--', lw=1.5, alpha=0.35,
             label='ideal  (α=0, β=0, γ=90°)')
    ax1.plot(t_arr, theta, color=col, lw=2.5,
             label='current' + ('  [✓ S-curve]' if ok else '  [✗ broken]'))
    ax1.axhline(0,  color='gray', lw=0.5)
    ax1.axhline(90, color='gray', lw=0.5, ls=':')
    ax1.axvline(time_val, color='orange', lw=1.5, ls='--', alpha=0.8)
    ax1.set_xlabel('time  t', fontsize=12)
    ax1.set_ylabel('θ(t)  [°]', fontsize=12)
    ax1.set_title('Joint angle trajectory', fontsize=12)
    ax1.legend(fontsize=10)
    ax1.set_xlim(0, 1)

    # right: arm drawing
    ax2.set_aspect('equal')
    ax2.set_xlim(-0.3, 1.35)
    ax2.set_ylim(-0.35, 1.3)
    ax2.axis('off')
    ax2.set_title(f'Arm at  t = {time_val:.2f}', fontsize=12)

    L = 1.0
    th_ideal_rad = np.deg2rad(theta_ideal)
    th_curr_rad  = np.deg2rad(theta)

    # tip-path traces
    ax2.plot(L*np.cos(th_ideal_rad), L*np.sin(th_ideal_rad),
             'k--', lw=1, alpha=0.22)
    ax2.plot(L*np.cos(th_curr_rad),  L*np.sin(th_curr_rad),
             color=col, lw=1.5, alpha=0.4)

    # current arm at selected time
    idx    = np.argmin(np.abs(t_arr - time_val))
    th     = np.deg2rad(theta[idx])
    ex, ey = L*np.cos(th), L*np.sin(th)

    # mount block
    rect = mpatches.FancyBboxPatch((-0.1, -0.12), 0.2, 0.12,
                                    boxstyle='round,pad=0.01',
                                    fc='#555', ec='#333', zorder=5)
    ax2.add_patch(rect)
    # arm body (two lines for outline + fill effect)
    ax2.plot([0, ex], [0, ey], color='#444', lw=7,
             solid_capstyle='round', zorder=3)
    ax2.plot([0, ex], [0, ey], color=col,   lw=5,
             solid_capstyle='round', zorder=4)
    # pivot + tip dots
    ax2.plot(0,  0,  'o', color='#333',   ms=9,  zorder=6)
    ax2.plot(ex, ey, 'o', color='orange', ms=10, zorder=6)

    # angle arc
    arc_a = np.linspace(0, max(th, 1e-6), 60)
    r_arc = 0.28
    ax2.plot(r_arc*np.cos(arc_a), r_arc*np.sin(arc_a),
             color='gray', lw=1.5)
    mid_a = th / 2
    ax2.text(0.44*np.cos(mid_a), 0.44*np.sin(mid_a),
             f'{theta[idx]:.1f}°', fontsize=9,
             ha='center', va='center', color='gray')

    # reference horizontal arrow
    ax2.annotate('', xy=(1.18, 0), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='lightgray', lw=1.2))

    plt.tight_layout()
    plt.show()


def redraw():
    theta = get_theta()
    with out:
        out.clear_output(wait=True)
        draw(theta, sl_time.value)
    c2, c3 = sl_c2.value, sl_c3.value
    resid  = 2*c2 + 3*c3
    if abs(resid) < 5:
        residual_html.value = (
            '<span style="color:green;font-size:13px">'
            '✓ S-curve intact  (2c₂ + 3c₃ = 0)</span>')
    else:
        residual_html.value = (
            f'<span style="color:tomato;font-size:13px">'
            f'✗ broken  (2c₂ + 3c₃ = {resid:.0f} ≠ 0)</span>')


def on_motion(change):
    global _busy
    if _busy: return
    _busy = True
    a, b, g = sl_alpha.value, sl_beta.value, sl_gamma.value
    for sl, v in [(sl_c0, a), (sl_c1, b), (sl_c2, 3*g), (sl_c3, -2*g)]:
        sl.value = float(np.clip(v, sl.min, sl.max))
    _busy = False
    redraw()


def on_mono(change):
    global _busy
    if _busy: return
    _busy = True
    c0, c1, c2, c3 = sl_c0.value, sl_c1.value, sl_c2.value, sl_c3.value
    g = (3*c2 - 2*c3) / 13       # least-squares projection onto S(t)
    for sl, v in [(sl_alpha, c0), (sl_beta, c1), (sl_gamma, g)]:
        sl.value = float(np.clip(v, sl.min, sl.max))
    _busy = False
    redraw()


sl_time.observe(lambda c: redraw(), names='value')
for sl in [sl_alpha, sl_beta, sl_gamma]:
    sl.observe(on_motion, names='value')
for sl in [sl_c0, sl_c1, sl_c2, sl_c3]:
    sl.observe(on_mono,   names='value')

# ── layout ─────────────────────────────────────────────────────────────────
left_panel = widgets.VBox([
    widgets.HTML(
        '<b style="font-size:14px;color:#2c5f8a">■ Motion basis  {1, t, S(t)}</b><br>'
        '<small style="color:#555">one knob = one physical property</small>'),
    sl_alpha, sl_beta, sl_gamma,
])
right_panel = widgets.VBox([
    widgets.HTML(
        '<b style="font-size:14px;color:#8a2c2c">■ Monomial basis  {1, t, t², t³}</b><br>'
        '<small style="color:#555">try to change only swing amplitude…</small>'),
    sl_c0, sl_c1, sl_c2, sl_c3,
    residual_html,
])
time_row = widgets.HBox([
    widgets.HTML('<b style="font-size:12px">Scrub arm position:</b>&nbsp;'),
    sl_time,
])

redraw()
display(out)
display(time_row)
display(widgets.HBox([left_panel, widgets.HTML('&emsp;' * 3), right_panel]))
